# Haridwar Wheat Phenology — S1/S2 Preprocessing, AI Benchmarking & Transition Analysis

This Colab notebook is the next-stage processing pipeline for the Haridwar KVK wheat experiment.

It performs:

1. Sentinel-2 QA and same-date tile deduplication.
2. Sentinel-1 orbit-track cleaning while keeping ASCENDING/DESCENDING geometries separate.
3. S1↔S2 calendar fusion using nearest observations from each relative orbit.
4. Phenology-label construction from the published stage-date anchors.
5. Leave-one-season-out (LOSO) AI-model benchmarking.
6. Accuracy, balanced accuracy, macro-F1, weighted-F1, Cohen's kappa, MCC, classification reports and confusion matrices.
7. S2-only vs S1-only vs S1+S2 fusion ablation.
8. Permutation feature importance.
9. Weighted Whittaker and double-logistic phenology curves.
10. SOS/POS/EOS interval-error evaluation.
11. Synthetic optical-gap robustness testing.

## Scientific guardrails

- There is one field and two seasons, so random acquisition-level train/test splits are not valid as the primary evaluation.
- LOSO is used: train 2023–24 → test 2024–25, then reverse.
- Date, DOY and days-from-window-start are never used as predictors.
- The primary classification target is a 5-group phenology target. A 10-stage target is kept only as exploratory.
- The daily labels are derived from published field-observed stage-date anchors; they are not raw daily ground-truth observations.
- Deep LSTM/TCN/Transformer models are not treated as primary valid benchmarks because there are only two independent seasonal sequences.
- Calendar dates are used directly; DAS is not recomputed from inconsistent sowing dates in the supplied scripts.

Reference paper: https://doi.org/10.1016/j.ecoinf.2026.103821


In [ ]:
!pip -q install xgboost lightgbm catboost


In [ ]:
from pathlib import Path
import warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.optimize import curve_fit

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_score, recall_score, classification_report,
    ConfusionMatrixDisplay, cohen_kappa_score, matthews_corrcoef
)
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    HistGradientBoostingClassifier, GradientBoostingClassifier
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUT = Path("/content/haridwar_outputs")
OUT.mkdir(parents=True, exist_ok=True)

S2_FILE = "/content/haridwar_s2_scene_features.csv"
S1_FILE = "/content/haridwar_s1_scene_features.csv"
SITE_FILE = "/content/haridwar_site_metadata.csv"

print("Output directory:", OUT)


In [ ]:
# Upload the 3 GEE exports when running in Colab.
from google.colab import files

needed = [S2_FILE, S1_FILE, SITE_FILE]
missing = [p for p in needed if not Path(p).exists()]

if missing:
    print("Upload:")
    print("  haridwar_s2_scene_features.csv")
    print("  haridwar_s1_scene_features.csv")
    print("  haridwar_site_metadata.csv")
    files.upload()

s2_raw = pd.read_csv(S2_FILE)
s1_raw = pd.read_csv(S1_FILE)
site = pd.read_csv(SITE_FILE)

print("S2 raw:", s2_raw.shape)
print("S1 raw:", s1_raw.shape)
print("Site:", site.shape)
display(site)


In [ ]:
# ---------------------------
# 1. DATA AUDIT
# ---------------------------
required_s2 = {
    "season","acquisition_date","cloud_free",
    "s2_clear_fraction","s2_cs_cdf_median",
    "NDVI","NDRE","GNDVI","SAVI","EVI","LSWI"
}
required_s1 = {
    "season","acquisition_date","orbit_pass",
    "relative_orbit_start","VV","VH","VV_minus_VH","RVI","ANGLE"
}

assert required_s2.issubset(s2_raw.columns), required_s2 - set(s2_raw.columns)
assert required_s1.issubset(s1_raw.columns), required_s1 - set(s1_raw.columns)

s2_raw["date"] = pd.to_datetime(s2_raw["acquisition_date"])
s1_raw["date"] = pd.to_datetime(s1_raw["acquisition_date"])

print("S2 rows by season")
display(s2_raw.groupby("season").size().rename("rows").to_frame())

print("S2 unique dates")
display(s2_raw.groupby("season")["date"].nunique().rename("unique_dates").to_frame())

print("S2 cloud flags")
display(pd.crosstab(s2_raw["season"], s2_raw["cloud_free"], margins=True))

print("S2 granules per season/date")
display(
    s2_raw.groupby(["season","date"]).size()
          .value_counts().sort_index()
          .rename("number_of_dates").to_frame()
)

print("S1 rows by season/pass")
display(s1_raw.groupby(["season","orbit_pass"]).size().rename("rows").to_frame())

print("S1 rows by season/pass/relative orbit")
display(
    s1_raw.groupby(["season","orbit_pass","relative_orbit_start"])
          .size().rename("rows").to_frame()
)


## Published phenology anchors

The notebook uses the published field-observed stage dates as calendar-date anchors.

For the primary classifier, the 10 stages are grouped into five states:

- Establishment: Germination + Leaf Development
- Vegetative: Tillering + Stem Elongation
- Reproductive: Booting + Heading + Flowering
- Grain Filling: Development of Fruit + Ripening
- Senescence/Harvest: Senescence/Harvest

To obtain labels for satellite acquisition dates, midpoint boundaries are placed between adjacent published stage anchors. These are called `derived_daily_stage` labels in the notebook.


In [ ]:
stage_rows = [
    ("wheat_2023_24","Germination","2023-11-04"),
    ("wheat_2023_24","Leaf Development","2023-11-25"),
    ("wheat_2023_24","Tillering","2023-12-30"),
    ("wheat_2023_24","Stem Elongation","2024-02-10"),
    ("wheat_2023_24","Booting","2024-03-03"),
    ("wheat_2023_24","Heading","2024-03-20"),
    ("wheat_2023_24","Flowering","2024-03-31"),
    ("wheat_2023_24","Development of Fruit","2024-04-06"),
    ("wheat_2023_24","Ripening","2024-04-12"),
    ("wheat_2023_24","Senescence/Harvest","2024-04-18"),

    ("wheat_2024_25","Germination","2024-12-26"),
    ("wheat_2024_25","Leaf Development","2025-01-09"),
    ("wheat_2024_25","Tillering","2025-01-29"),
    ("wheat_2024_25","Stem Elongation","2025-02-22"),
    ("wheat_2024_25","Booting","2025-03-16"),
    ("wheat_2024_25","Heading","2025-03-30"),
    ("wheat_2024_25","Flowering","2025-04-09"),
    ("wheat_2024_25","Development of Fruit","2025-04-16"),
    ("wheat_2024_25","Ripening","2025-04-22"),
    ("wheat_2024_25","Senescence/Harvest","2025-04-26"),
]

anchors = pd.DataFrame(stage_rows, columns=["season","observed_stage","observed_date"])
anchors["observed_date"] = pd.to_datetime(anchors["observed_date"])

GROUP_MAP = {
    "Germination":"Establishment",
    "Leaf Development":"Establishment",
    "Tillering":"Vegetative",
    "Stem Elongation":"Vegetative",
    "Booting":"Reproductive",
    "Heading":"Reproductive",
    "Flowering":"Reproductive",
    "Development of Fruit":"Grain Filling",
    "Ripening":"Grain Filling",
    "Senescence/Harvest":"Senescence/Harvest"
}

anchors["stage_group_5"] = anchors["observed_stage"].map(GROUP_MAP)
display(anchors)
anchors.to_csv(OUT / "published_stage_anchors.csv", index=False)


In [ ]:
# ---------------------------
# 2. S2 SAME-DATE DEDUPLICATION
# ---------------------------
S2_SCIENCE = [
    "B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12",
    "GCC","NDVI","NDRE","GNDVI","SAVI","EVI","LSWI"
]

S2_QA = [
    "s2_valid_fraction","s2_clear_fraction","s2_cs_cdf_median",
    "clear_pixel_count","cloudy_pixel_percentage"
]

for c in S2_SCIENCE + S2_QA:
    s2_raw[c] = pd.to_numeric(s2_raw[c], errors="coerce")

agg_map = {c:"median" for c in S2_SCIENCE}
agg_map.update({
    "s2_valid_fraction":"max",
    "s2_clear_fraction":"max",
    "s2_cs_cdf_median":"median",
    "clear_pixel_count":"max",
    "cloudy_pixel_percentage":"median"
})

s2_daily = (
    s2_raw.groupby(["season","date"], as_index=False)
          .agg(agg_map)
)

counts = (
    s2_raw.groupby(["season","date"])
          .agg(
              n_s2_granules=("scene_id","size"),
              n_valid_optical_granules=("NDVI", lambda x: x.notna().sum()),
              cloud_free_any=("cloud_free", lambda x: int((x.astype(str)=="Y").any()))
          )
          .reset_index()
)

s2_daily = (
    s2_daily.merge(counts, on=["season","date"], how="left")
            .sort_values(["season","date"])
            .reset_index(drop=True)
)

print("Raw S2 rows:", len(s2_raw))
print("One-row-per-date S2:", len(s2_daily))
display(s2_daily.groupby("season").size().rename("unique_dates").to_frame())

s2_daily.to_csv(OUT / "haridwar_s2_daily_collapsed.csv", index=False)


In [ ]:
# ---------------------------
# 3. S1 ORBIT-TRACK CLEANING
# ---------------------------
for c in ["relative_orbit_start","VV","VH","VV_minus_VH","RVI","ANGLE","valid_pixel_count"]:
    s1_raw[c] = pd.to_numeric(s1_raw[c], errors="coerce")

s1_raw["relative_orbit_start"] = s1_raw["relative_orbit_start"].round().astype("Int64")
s1_raw["track_id"] = (
    s1_raw["orbit_pass"].str[0].str.upper()
    + s1_raw["relative_orbit_start"].astype(str)
)

S1_SCIENCE = ["VV","VH","VV_minus_VH","RVI"]

s1_tracks = (
    s1_raw.groupby(["season","date","track_id"], as_index=False)
          .agg({
              "VV":"median",
              "VH":"median",
              "VV_minus_VH":"median",
              "RVI":"median",
              "ANGLE":"median",
              "valid_pixel_count":"max"
          })
          .sort_values(["season","date","track_id"])
          .reset_index(drop=True)
)

print("Track IDs:", sorted(s1_tracks["track_id"].dropna().unique()))
display(
    s1_tracks.groupby(["season","track_id"]).size()
             .rename("observations").to_frame()
)

s1_tracks.to_csv(OUT / "haridwar_s1_tracks_clean.csv", index=False)


In [ ]:
# ---------------------------
# 4. FUSE EACH S1 TRACK ONTO THE S2 CALENDAR
# ---------------------------
# Individual S1 relative orbits repeat at roughly 12-day cadence.
# ±6 days avoids mixing arbitrarily distant SAR observations.
S1_TOLERANCE_DAYS = 6

fusion = s2_daily.copy()
track_ids = sorted(s1_tracks["track_id"].dropna().unique())

for track in track_ids:
    prefix = f"S1_{track}_"
    merged_seasons = []

    for season, left in fusion.groupby("season", sort=False):
        left = left.sort_values("date").copy()

        right = (
            s1_tracks[
                (s1_tracks["season"] == season) &
                (s1_tracks["track_id"] == track)
            ]
            .sort_values("date")
            .copy()
        )

        if right.empty:
            for c in S1_SCIENCE + ["ANGLE"]:
                left[prefix + c] = np.nan
            left[prefix + "delta_days"] = np.nan
            merged_seasons.append(left)
            continue

        right = right[["date"] + S1_SCIENCE + ["ANGLE"]].copy()
        right[prefix + "s1_date"] = right["date"]

        right = right.rename(
            columns={c:prefix+c for c in S1_SCIENCE + ["ANGLE"]}
        )

        m = pd.merge_asof(
            left,
            right,
            on="date",
            direction="nearest",
            tolerance=pd.Timedelta(days=S1_TOLERANCE_DAYS)
        )

        m[prefix + "delta_days"] = (
            m["date"] - m[prefix + "s1_date"]
        ).dt.total_seconds().abs() / 86400.0

        m = m.drop(columns=[prefix + "s1_date"])
        merged_seasons.append(m)

    fusion = (
        pd.concat(merged_seasons, ignore_index=True)
          .sort_values(["season","date"])
          .reset_index(drop=True)
    )

sar_cols = [c for c in fusion.columns if c.startswith("S1_")]

print("Fusion table:", fusion.shape)
print("SAR feature coverage:")
display(fusion[sar_cols].notna().mean().sort_values().rename("coverage").to_frame())

fusion.to_csv(OUT / "haridwar_s2_s1_fusion_calendar.csv", index=False)


In [ ]:
# ---------------------------
# 5. DERIVED DAILY PHENOLOGY LABELS
# ---------------------------
def stage_labels_for_dates(dates, season):
    a = (
        anchors[anchors["season"] == season]
        .sort_values("observed_date")
        .reset_index(drop=True)
    )

    ad = a["observed_date"].values.astype("datetime64[ns]")
    boundaries = ad[:-1] + (ad[1:] - ad[:-1]) / 2

    x = pd.to_datetime(dates).values.astype("datetime64[ns]")
    idx = np.searchsorted(boundaries, x, side="right")
    idx = np.clip(idx, 0, len(a)-1)

    stages = a.loc[idx, "observed_stage"].to_numpy()
    groups = pd.Series(stages).map(GROUP_MAP).to_numpy()
    return stages, groups

fusion["derived_daily_stage"] = None
fusion["stage_group_5"] = None
fusion["within_observed_phenology_window"] = False

for season in fusion["season"].unique():
    a = anchors[anchors["season"] == season].sort_values("observed_date")
    lo = a["observed_date"].min()
    hi = a["observed_date"].max()

    mask = (fusion["season"] == season) & fusion["date"].between(lo, hi)
    stages, groups = stage_labels_for_dates(fusion.loc[mask, "date"], season)

    fusion.loc[mask, "derived_daily_stage"] = stages
    fusion.loc[mask, "stage_group_5"] = groups
    fusion.loc[mask, "within_observed_phenology_window"] = True

model_df = (
    fusion[fusion["within_observed_phenology_window"]]
    .copy()
    .reset_index(drop=True)
)

print("Rows inside observed phenology windows:", len(model_df))

print("5-group support")
display(pd.crosstab(model_df["season"], model_df["stage_group_5"]))

print("10-stage support")
display(pd.crosstab(model_df["season"], model_df["derived_daily_stage"]))

model_df.to_csv(OUT / "haridwar_model_dataset.csv", index=False)


In [ ]:
# ---------------------------
# 6. VISUAL QA
# ---------------------------
fig, ax = plt.subplots(figsize=(13,5))

for season, g in s2_daily.groupby("season"):
    ax.plot(g["date"], g["NDVI"], marker="o", label=season)

for _, r in anchors.iterrows():
    ax.axvline(r["observed_date"], alpha=0.12)

ax.set_title("Sentinel-2 NDVI with published phenology anchors")
ax.set_ylabel("NDVI")
ax.set_xlabel("Date")
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "01_s2_ndvi_stage_anchors.png", dpi=180)
plt.show()


## AI benchmark

Primary evaluation is leave-one-season-out.

Compared model families:

- Logistic Regression
- RBF SVM
- k-NN
- Gaussian Naive Bayes
- Random Forest
- Extra Trees
- Gradient Boosting
- HistGradientBoosting
- XGBoost
- LightGBM
- CatBoost

Three sensor configurations are evaluated on the same S2-date calendar:

- S2-only
- S1-only
- S1+S2 fusion

The preprocessing pipeline is fit separately inside each training fold. Missingness indicators are added automatically.


In [ ]:
# ---------------------------
# 7. FEATURE SETS
# ---------------------------
S2_MODEL_FEATURES = S2_SCIENCE + [
    "s2_valid_fraction",
    "s2_clear_fraction",
    "s2_cs_cdf_median",
    "clear_pixel_count",
    "cloudy_pixel_percentage"
]

S1_MODEL_FEATURES = [
    c for c in model_df.columns
    if c.startswith("S1_") and not c.endswith("_ANGLE")
]

FUSION_FEATURES = S2_MODEL_FEATURES + S1_MODEL_FEATURES

FEATURE_SETS = {
    "S2_only": S2_MODEL_FEATURES,
    "S1_only": S1_MODEL_FEATURES,
    "S1_S2_fusion": FUSION_FEATURES
}

for k,v in FEATURE_SETS.items():
    print(k, len(v), "features")


In [ ]:
# ---------------------------
# 8. MODEL REGISTRY
# ---------------------------
def make_models():
    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "SVM_RBF": SVC(
            C=3.0,
            gamma="scale",
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ),
        "GaussianNB": GaussianNB(),
        "RandomForest": RandomForestClassifier(
            n_estimators=500,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=500,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.03,
            max_depth=2,
            random_state=RANDOM_STATE
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=250,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    }

    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = XGBClassifier(
            n_estimators=350,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    except Exception as e:
        print("XGBoost skipped:", e)

    try:
        from lightgbm import LGBMClassifier
        models["LightGBM"] = LGBMClassifier(
            n_estimators=350,
            learning_rate=0.03,
            num_leaves=15,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            verbosity=-1
        )
    except Exception as e:
        print("LightGBM skipped:", e)

    try:
        from catboost import CatBoostClassifier
        models["CatBoost"] = CatBoostClassifier(
            iterations=350,
            depth=5,
            learning_rate=0.03,
            loss_function="MultiClass",
            auto_class_weights="Balanced",
            verbose=False,
            random_seed=RANDOM_STATE
        )
    except Exception as e:
        print("CatBoost skipped:", e)

    return models

MODELS = make_models()
print("Models loaded:", list(MODELS))


In [ ]:
# ---------------------------
# 9. LOSO BENCHMARK
# ---------------------------
TARGET = "stage_group_5"

label_encoder = LabelEncoder()
y_all = label_encoder.fit_transform(model_df[TARGET].astype(str))
CLASS_NAMES = list(label_encoder.classes_)

def make_pipeline(estimator):
    return Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True
        )),
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])

def metric_row(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred)
    }

rows = []
pred_rows = []
seasons = sorted(model_df["season"].unique())

for fs_name, features in FEATURE_SETS.items():
    X = model_df[features].apply(pd.to_numeric, errors="coerce")

    for model_name, estimator in MODELS.items():
        oof_true = []
        oof_pred = []

        for test_season in seasons:
            train_mask = model_df["season"] != test_season
            test_mask = model_df["season"] == test_season

            X_train = X.loc[train_mask]
            X_test = X.loc[test_mask]
            y_train = y_all[train_mask.to_numpy()]
            y_test = y_all[test_mask.to_numpy()]

            if len(np.unique(y_train)) < 2:
                continue

            pipe = make_pipeline(clone(estimator))

            try:
                pipe.fit(X_train, y_train)
                pred = pipe.predict(X_test)
            except Exception as e:
                print("SKIP:", fs_name, model_name, test_season, str(e)[:120])
                continue

            rows.append({
                "feature_set":fs_name,
                "model":model_name,
                "test_season":test_season,
                **metric_row(y_test, pred)
            })

            oof_true.extend(y_test.tolist())
            oof_pred.extend(pred.tolist())

            true_names = label_encoder.inverse_transform(y_test)
            pred_names = label_encoder.inverse_transform(pred)

            for idx, yt, yp in zip(model_df.index[test_mask], true_names, pred_names):
                pred_rows.append({
                    "row_index":idx,
                    "date":model_df.loc[idx,"date"],
                    "season":model_df.loc[idx,"season"],
                    "feature_set":fs_name,
                    "model":model_name,
                    "y_true":yt,
                    "y_pred":yp
                })

        if oof_true:
            rows.append({
                "feature_set":fs_name,
                "model":model_name,
                "test_season":"LOSO_OOF",
                **metric_row(np.array(oof_true), np.array(oof_pred))
            })

results = pd.DataFrame(rows)
predictions = pd.DataFrame(pred_rows)

summary = (
    results[results["test_season"]=="LOSO_OOF"]
    .sort_values(["feature_set","macro_f1"], ascending=[True,False])
    .reset_index(drop=True)
)

display(summary)

results.to_csv(OUT / "model_metrics_all_folds.csv", index=False)
summary.to_csv(OUT / "model_metrics_loso_oof.csv", index=False)
predictions.to_csv(OUT / "model_oof_predictions.csv", index=False)


In [ ]:
# ---------------------------
# 10. MODEL COMPARISON PLOTS
# ---------------------------
for fs_name in FEATURE_SETS:
    d = summary[summary["feature_set"]==fs_name].sort_values("macro_f1")
    if d.empty:
        continue

    fig, ax = plt.subplots(figsize=(9,5))
    ax.barh(d["model"], d["macro_f1"])
    ax.set_xlim(0,1)
    ax.set_xlabel("LOSO macro-F1")
    ax.set_title(f"Model comparison — {fs_name}")
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.savefig(OUT / f"02_model_comparison_{fs_name}.png", dpi=180)
    plt.show()


In [ ]:
# ---------------------------
# 11. CONFUSION MATRICES
# ---------------------------
def plot_confusions(feature_set, top_n=3):
    top = (
        summary[summary["feature_set"]==feature_set]
        .nlargest(top_n, "macro_f1")
    )

    for _, r in top.iterrows():
        model_name = r["model"]

        p = predictions[
            (predictions["feature_set"]==feature_set) &
            (predictions["model"]==model_name)
        ].copy()

        if p.empty:
            continue

        fig, ax = plt.subplots(figsize=(7,6))
        ConfusionMatrixDisplay.from_predictions(
            p["y_true"], p["y_pred"],
            labels=CLASS_NAMES,
            display_labels=CLASS_NAMES,
            xticks_rotation=45,
            cmap="Blues",
            colorbar=False,
            ax=ax
        )
        ax.set_title(f"{feature_set} — {model_name}\nLOSO confusion matrix")
        plt.tight_layout()
        plt.savefig(
            OUT / f"03_cm_counts_{feature_set}_{model_name}.png",
            dpi=180
        )
        plt.show()

        fig, ax = plt.subplots(figsize=(7,6))
        ConfusionMatrixDisplay.from_predictions(
            p["y_true"], p["y_pred"],
            labels=CLASS_NAMES,
            display_labels=CLASS_NAMES,
            normalize="true",
            values_format=".2f",
            xticks_rotation=45,
            cmap="Blues",
            colorbar=False,
            ax=ax
        )
        ax.set_title(f"{feature_set} — {model_name}\nNormalized LOSO confusion matrix")
        plt.tight_layout()
        plt.savefig(
            OUT / f"04_cm_normalized_{feature_set}_{model_name}.png",
            dpi=180
        )
        plt.show()

for fs in FEATURE_SETS:
    plot_confusions(fs, top_n=3)


In [ ]:
# ---------------------------
# 12. CLASSIFICATION REPORTS
# ---------------------------
for fs_name in FEATURE_SETS:
    d = summary[summary["feature_set"]==fs_name]
    if d.empty:
        continue

    leader = d.iloc[0]["model"]
    p = predictions[
        (predictions["feature_set"]==fs_name) &
        (predictions["model"]==leader)
    ]

    report = classification_report(
        p["y_true"],
        p["y_pred"],
        labels=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report).T

    print("\n", "="*75)
    print(fs_name, "leader:", leader)
    display(report_df)

    report_df.to_csv(
        OUT / f"classification_report_{fs_name}_{leader}.csv"
    )

print("\nPer-season metrics")
display(
    results[results["test_season"]!="LOSO_OOF"][
        ["feature_set","model","test_season",
         "macro_f1","balanced_accuracy","accuracy","kappa","mcc"]
    ].sort_values(["feature_set","model","test_season"])
)


In [ ]:
# ---------------------------
# 13. PERMUTATION IMPORTANCE
# ---------------------------
fusion_rank = summary[summary["feature_set"]=="S1_S2_fusion"]

if not fusion_rank.empty:
    leader_name = fusion_rank.iloc[0]["model"]
    leader_estimator = MODELS[leader_name]
    features = FEATURE_SETS["S1_S2_fusion"]

    importance_rows = []

    for test_season in seasons:
        train_mask = model_df["season"] != test_season
        test_mask = model_df["season"] == test_season

        X_train = model_df.loc[train_mask, features].apply(pd.to_numeric, errors="coerce")
        X_test = model_df.loc[test_mask, features].apply(pd.to_numeric, errors="coerce")
        y_train = y_all[train_mask.to_numpy()]
        y_test = y_all[test_mask.to_numpy()]

        pipe = make_pipeline(clone(leader_estimator))
        pipe.fit(X_train, y_train)

        pi = permutation_importance(
            pipe,
            X_test,
            y_test,
            scoring="f1_macro",
            n_repeats=30,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        for f, mean, std in zip(features, pi.importances_mean, pi.importances_std):
            importance_rows.append({
                "test_season":test_season,
                "feature":f,
                "importance_mean":mean,
                "importance_std":std
            })

    importance = pd.DataFrame(importance_rows)

    importance_summary = (
        importance.groupby("feature")["importance_mean"]
                  .mean()
                  .sort_values(ascending=False)
                  .head(20)
    )

    display(importance_summary.to_frame())

    fig, ax = plt.subplots(figsize=(9,7))
    importance_summary.sort_values().plot.barh(ax=ax)
    ax.set_xlabel("Mean decrease in macro-F1")
    ax.set_title(f"Fusion permutation importance — {leader_name}")
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.savefig(OUT / "05_fusion_permutation_importance.png", dpi=180)
    plt.show()

    importance.to_csv(
        OUT / "fusion_permutation_importance_all_folds.csv",
        index=False
    )


## 10-stage target support check

The exact 10-stage target is not the primary result. Before attempting it, inspect whether both seasons contain enough satellite observations in every stage. If any class is absent or represented by only one observation in a season, do not use a 10-class confusion matrix as the main paper result.


In [ ]:
support10 = pd.crosstab(
    model_df["season"],
    model_df["derived_daily_stage"]
)
display(support10)

rare = support10.columns[(support10.min(axis=0) < 2)]
if len(rare):
    print("10-stage classes with <2 observations in at least one season:")
    print(list(rare))
    print("Keep 10-stage analysis exploratory; use the 5-group target as primary.")
else:
    print("All 10 stages have >=2 samples in both seasons.")


## Phenology curve analysis

Two optical curve methods are evaluated:

- **Weighted Whittaker smoother** — handles missing S2 data explicitly.
- **Double logistic fit** — conventional phenology baseline.

Transition definitions from the fitted/smoothed NDVI curve:

- SOS: first 20% seasonal-amplitude crossing on the rising limb
- POS: maximum NDVI
- EOS: last 20% seasonal-amplitude crossing on the falling limb

Reference intervals:

- SOS: Germination → Tillering
- POS: Heading → Flowering
- EOS: Ripening → Senescence/Harvest

The error is zero when a predicted transition falls inside its reference interval; otherwise it is the distance in days to the nearest interval boundary.


In [ ]:
# ---------------------------
# 14. WHITTAKER + DOUBLE LOGISTIC
# ---------------------------
def whittaker(y, weights=None, lam=100.0):
    y = np.asarray(y, dtype=float)
    n = len(y)

    if weights is None:
        weights = np.isfinite(y).astype(float)
    else:
        weights = np.asarray(weights, dtype=float)
        weights = weights * np.isfinite(y)

    yy = np.nan_to_num(y, nan=0.0)

    D = sparse.diags(
        [
            np.ones(n-2),
            -2*np.ones(n-2),
            np.ones(n-2)
        ],
        [0,1,2],
        shape=(n-2,n),
        format="csc"
    )

    W = sparse.diags(weights, format="csc")
    A = W + lam * (D.T @ D)

    return np.asarray(spsolve(A, weights * yy))


def double_logistic(t, c, a, b, r1, r2, t1, t2):
    t = np.asarray(t, dtype=float)
    return (
        c
        + a / (1.0 + np.exp(-r1 * (t-t1)))
        - b / (1.0 + np.exp(-r2 * (t-t2)))
    )


def fit_double_logistic(dates, values):
    dates = pd.to_datetime(dates)
    values = np.asarray(values, dtype=float)

    good = np.isfinite(values)
    if good.sum() < 8:
        return None

    t0 = dates.min()
    t = (dates - t0).days.to_numpy(dtype=float)
    tg = t[good]
    yg = values[good]

    ylo = np.nanpercentile(yg, 10)
    yhi = np.nanpercentile(yg, 90)
    amp = max(yhi-ylo, 0.05)

    p0 = [
        ylo, amp, amp,
        0.08, 0.08,
        np.nanpercentile(tg, 30),
        np.nanpercentile(tg, 75)
    ]

    lower = [-0.5,0,0,0.001,0.001,0,0]
    upper = [1.0,2,2,1,1,t.max(),t.max()]

    try:
        pars, _ = curve_fit(
            double_logistic,
            tg, yg,
            p0=p0,
            bounds=(lower, upper),
            maxfev=50000
        )
    except Exception:
        return None

    curve = double_logistic(t, *pars)
    return {
        "dates":dates,
        "curve":curve,
        "params":pars
    }


def transition_dates_from_curve(dates, curve, fraction=0.20):
    dates = pd.to_datetime(dates)
    curve = np.asarray(curve, dtype=float)

    peak_i = int(np.nanargmax(curve))
    low = np.nanmin(curve)
    peak = curve[peak_i]
    thr = low + fraction * (peak-low)

    rising = np.where(curve[:peak_i+1] >= thr)[0]
    falling = np.where(curve[peak_i:] >= thr)[0]

    sos_i = int(rising[0]) if len(rising) else 0
    eos_i = int(peak_i + falling[-1]) if len(falling) else len(curve)-1

    return {
        "SOS":dates[sos_i],
        "POS":dates[peak_i],
        "EOS":dates[eos_i]
    }


def reference_intervals(season):
    a = (
        anchors[anchors["season"]==season]
        .set_index("observed_stage")["observed_date"]
    )

    return {
        "SOS":(a["Germination"], a["Tillering"]),
        "POS":(a["Heading"], a["Flowering"]),
        "EOS":(a["Ripening"], a["Senescence/Harvest"])
    }


def interval_error_days(pred, interval):
    L, U = interval
    pred = pd.Timestamp(pred)

    if L <= pred <= U:
        return 0.0
    if pred < L:
        return float((L-pred).days)
    return float((pred-U).days)


In [ ]:
# ---------------------------
# 15. RUN TRANSITION EVALUATION
# ---------------------------
curve_rows = []
transition_rows = []

for season in sorted(s2_daily["season"].unique()):
    g = (
        s2_daily[s2_daily["season"]==season]
        .sort_values("date")
        .copy()
    )

    a = anchors[anchors["season"]==season]
    lo = a["observed_date"].min()
    hi = a["observed_date"].max()

    daily = pd.DataFrame({
        "date":pd.date_range(lo, hi, freq="D")
    }).merge(
        g[["date","NDVI","s2_clear_fraction","s2_cs_cdf_median"]],
        on="date",
        how="left"
    )

    y = daily["NDVI"].to_numpy(float)

    weights = (
        daily["s2_clear_fraction"].fillna(0).clip(0,1)
        * daily["s2_cs_cdf_median"].fillna(0).clip(0,1)
    ).to_numpy(float)

    weights[~np.isfinite(y)] = 0

    z_wh = whittaker(y, weights, lam=100.0)
    tr_wh = transition_dates_from_curve(daily["date"], z_wh)

    dl = fit_double_logistic(daily["date"], y)
    tr_dl = None if dl is None else transition_dates_from_curve(
        dl["dates"], dl["curve"]
    )

    for d, raw, smooth in zip(daily["date"], y, z_wh):
        curve_rows.append({
            "season":season,
            "date":d,
            "NDVI_raw":raw,
            "NDVI_whittaker":smooth
        })

    refs = reference_intervals(season)

    for method, transitions in [
        ("Whittaker", tr_wh),
        ("DoubleLogistic", tr_dl)
    ]:
        if transitions is None:
            continue

        for key, pred in transitions.items():
            L,U = refs[key]

            transition_rows.append({
                "season":season,
                "method":method,
                "transition":key,
                "predicted_date":pred,
                "reference_start":L,
                "reference_end":U,
                "interval_error_days":interval_error_days(pred, (L,U))
            })

curves = pd.DataFrame(curve_rows)
transition_eval = pd.DataFrame(transition_rows)

transition_eval["within_7d"] = transition_eval["interval_error_days"] <= 7
transition_eval["within_14d"] = transition_eval["interval_error_days"] <= 14

display(transition_eval)

print("Mean interval error")
display(
    transition_eval.groupby(["method","transition"])["interval_error_days"]
                   .mean().unstack()
)

curves.to_csv(OUT / "smoothed_ndvi_curves.csv", index=False)
transition_eval.to_csv(OUT / "transition_date_evaluation.csv", index=False)


In [ ]:
# ---------------------------
# 16. PLOT SMOOTHED CURVES
# ---------------------------
for season in sorted(curves["season"].unique()):
    c = curves[curves["season"]==season]
    a = anchors[anchors["season"]==season]

    fig, ax = plt.subplots(figsize=(13,5))
    ax.scatter(
        c["date"], c["NDVI_raw"],
        s=28, label="Observed S2 NDVI"
    )
    ax.plot(
        c["date"], c["NDVI_whittaker"],
        linewidth=2.2, label="Weighted Whittaker"
    )

    for _, r in a.iterrows():
        ax.axvline(r["observed_date"], alpha=0.15)

    ax.set_title(f"{season}: S2 NDVI + published stage anchors")
    ax.set_ylabel("NDVI")
    ax.set_xlabel("Date")
    ax.grid(alpha=0.2)
    ax.legend()

    plt.tight_layout()
    plt.savefig(OUT / f"06_whittaker_{season}.png", dpi=180)
    plt.show()


## Synthetic optical-gap robustness

This directly tests whether SAR helps when optical observations are unavailable.

For each LOSO test season, 0%, 20%, 40% and 60% of test dates have their S2 predictors masked. The already-trained S2-only and fusion pipelines are then evaluated.

The model used for each feature set is the highest-macro-F1 model in the fixed benchmark. Therefore this section is exploratory model-comparison analysis; do not use it to claim an independently selected final model.


In [ ]:
# ---------------------------
# 17. OPTICAL-GAP ROBUSTNESS
# ---------------------------
def evaluate_with_optical_mask(
    feature_set,
    model_name,
    mask_fraction,
    repeats=25
):
    features = FEATURE_SETS[feature_set]
    estimator = MODELS[model_name]
    rows = []

    for test_season in seasons:
        train_mask = model_df["season"] != test_season
        test_mask = model_df["season"] == test_season

        X_train = (
            model_df.loc[train_mask, features]
            .apply(pd.to_numeric, errors="coerce")
        )
        X_test_base = (
            model_df.loc[test_mask, features]
            .apply(pd.to_numeric, errors="coerce")
        )

        y_train = y_all[train_mask.to_numpy()]
        y_test = y_all[test_mask.to_numpy()]

        pipe = make_pipeline(clone(estimator))
        pipe.fit(X_train, y_train)

        rng = np.random.default_rng(RANDOM_STATE)

        for rep in range(repeats):
            X_test = X_test_base.copy()
            nmask = int(round(mask_fraction * len(X_test)))

            if nmask > 0:
                mask_idx = rng.choice(
                    X_test.index,
                    size=nmask,
                    replace=False
                )

                optical_cols = [
                    c for c in S2_MODEL_FEATURES
                    if c in X_test.columns
                ]

                X_test.loc[mask_idx, optical_cols] = np.nan

            pred = pipe.predict(X_test)

            rows.append({
                "feature_set":feature_set,
                "model":model_name,
                "test_season":test_season,
                "mask_fraction":mask_fraction,
                "repeat":rep,
                "macro_f1":f1_score(
                    y_test, pred,
                    average="macro",
                    zero_division=0
                ),
                "balanced_accuracy":balanced_accuracy_score(
                    y_test, pred
                )
            })

    return pd.DataFrame(rows)


robust_parts = []

for fs in ["S2_only","S1_S2_fusion"]:
    rank = summary[summary["feature_set"]==fs]

    if rank.empty:
        continue

    leader = rank.iloc[0]["model"]

    for frac in [0.0,0.2,0.4,0.6]:
        robust_parts.append(
            evaluate_with_optical_mask(
                fs, leader, frac, repeats=25
            )
        )

robustness = pd.concat(robust_parts, ignore_index=True)

display(
    robustness.groupby(["feature_set","mask_fraction"])["macro_f1"]
              .agg(["mean","std"])
)

robustness.to_csv(
    OUT / "optical_gap_robustness.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(8,5))

for fs, g in robustness.groupby("feature_set"):
    s = (
        g.groupby("mask_fraction")["macro_f1"]
         .agg(["mean","std"])
         .reset_index()
    )

    ax.errorbar(
        s["mask_fraction"],
        s["mean"],
        yerr=s["std"],
        marker="o",
        label=fs
    )

ax.set_xlabel("Fraction of test dates with S2 predictors masked")
ax.set_ylabel("Macro-F1")
ax.set_ylim(0,1)
ax.set_title("Synthetic optical-gap robustness")
ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()
plt.savefig(OUT / "07_optical_gap_robustness.png", dpi=180)
plt.show()


In [ ]:
# ---------------------------
# 18. NATURAL CLOUD-GAP AUDIT
# ---------------------------
gap_rows = []

for season, g in s2_daily.groupby("season"):
    g = g.sort_values("date")
    usable_dates = g.loc[g["NDVI"].notna(), "date"].sort_values()

    max_clear_gap = (
        usable_dates.diff().dt.days.max()
        if len(usable_dates) >= 2
        else np.nan
    )

    gap_rows.append({
        "season":season,
        "n_s2_dates":len(g),
        "n_usable_optical_dates":int(g["NDVI"].notna().sum()),
        "usable_fraction":float(g["NDVI"].notna().mean()),
        "max_gap_between_usable_s2_dates_days":max_clear_gap
    })

natural_gaps = pd.DataFrame(gap_rows)
display(natural_gaps)

natural_gaps.to_csv(
    OUT / "natural_cloud_gap_summary.csv",
    index=False
)


## Recommended ETET reporting hierarchy

### Primary classification result
- 5-group phenology classification
- LOSO validation
- macro-F1 primary metric
- balanced accuracy secondary
- normalized confusion matrix
- per-season metrics

### Sensor ablation
- S2-only
- S1-only
- S1+S2 fusion

### Robustness
- natural cloud-gap statistics
- synthetic 20/40/60% optical masking
- compare degradation of S2-only and S1+S2 fusion

### Phenology-transition result
- weighted Whittaker vs double logistic
- SOS/POS/EOS interval error
- ±7-day and ±14-day agreement

### Do not claim
- derived daily stage labels are raw daily observations;
- two seasons demonstrate broad geographic generalization;
- random acquisition-level train/test splits are independent;
- deep-learning superiority from this sample size.


In [ ]:
# ---------------------------
# 19. SAVE + ZIP ALL RESULTS
# ---------------------------
print("Generated outputs:")
for p in sorted(OUT.iterdir()):
    print(" -", p.name)

zip_path = Path("/content/haridwar_outputs.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in OUT.iterdir():
        if p.is_file():
            z.write(p, arcname=p.name)

print("\nZIP:", zip_path)
print("Uncomment the next line to download it:")
print("# files.download(str(zip_path))")

# files.download(str(zip_path))
